# COMP34812 NLI Shared Task: Demo Code (Inference)
**Group Number:** CW47
**Track:** A. Natural Language Inference (NLI)
**Solution** Category C - Deep Learning (Transformer)
**Group Members:**
1. Munir Emre Tanats - 10959764

## Instructions for the Marker
This notebook runs our trained models in "Inference Mode" on the hidden test dataset.
1. Please ensure you have downloaded our trained model folder (`modernbert-large-rte-final`) from the OneDrive link provided in the `README`.
2. Upload the model folder and your hidden `test.csv` to this environment.
3. Update the `test_df` and `model_dir` variables in the code below to point to your uploaded files.
4. Run all cells to generate the prediction outputs.

---
## Solution 1: Transformer-Based Deep Learning (Category C)
**Model:** Fine-tuned `answerdotai/ModernBERT-Large`

This cell loads our fine-tuned ModernBERT-Large model from the specified directory, processes the premise-hypothesis pairs from the test file, and outputs the binary predictions to a CSV file.

In [4]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline



model_dir = "./modernbert-large-rte-final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

test_df = pd.read_csv("test.csv")

test_df['label'] = 0
test_dataset = Dataset.from_pandas(test_df)

def preprocess_function(examples):
    texts = [f"{p} [SEP] {h}" for p, h in zip(examples['premise'], examples['hypothesis'])]
    return tokenizer(texts, truncation=True, max_length=256, padding="max_length")

tokenized_test = test_dataset.map(preprocess_function, batched=True)

inference_args = TrainingArguments(
    output_dir="./temp_inference",
    per_device_eval_batch_size=8,
    fp16=True
)
trainer = Trainer(
    model=model,
    args=inference_args,
    processing_class=tokenizer
)

test_outputs = trainer.predict(tokenized_test)


probabilities = torch.nn.functional.softmax(torch.tensor(test_outputs.predictions), dim=-1).numpy()
prob_entailment = probabilities[:, 1]

OPTIMAL_THRESHOLD = 0.50

test_predictions = (prob_entailment >= OPTIMAL_THRESHOLD).astype(int)

final_df = pd.DataFrame({'prediction': test_predictions})

output_filename = "Group_CW47_NLI_C.csv"
final_df.to_csv(output_filename, index=False)

print(f"Final saved to {output_filename}")

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Map:   0%|          | 0/3302 [00:00<?, ? examples/s]

Final saved to Group_CW47_NLI_C.csv
